In [1]:
# 0. Setup: import libraries and define API key

import requests
import pandas as pd

API_KEY = "c0d18a0f217bfb86aeb4a9b089540c9a77a34f5b"

In [2]:
# 1. Constants (dataset configuration)

STATE_FIPS_CA = "06"
ACS_YEAR = 2024
ACS_BASE = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"

# Start with one variable (total population)
VARS = ["NAME", "B01003_001E"]

In [3]:
# 2. Pull all California places (sanity test)

params = {
    "get": ",".join(VARS),                     # Variables to retrieve
    "for": "place:*",                          # All places (cities + CDPs)
    "in": f"state:{STATE_FIPS_CA}",            # California (FIPS 06)
    "key": API_KEY                             # Census API key
}

response = requests.get(ACS_BASE, params=params)
response.raise_for_status()

data = response.json()

ca_places = pd.DataFrame(data[1:], columns=data[0])

# Convert population column to numeric
ca_places["B01003_001E"] = pd.to_numeric(
    ca_places["B01003_001E"], errors="coerce"
)

ca_places.head()

,NAME,B01003_001E,state,place
0,"Acalanes Ridge CDP, California",1006,06,00135
1,"Acampo CDP, California",234,06,00156
2,"Acton CDP, California",7880,06,00212
3,"Adelanto city, California",37964,06,00296
4,"Adin CDP, California",153,06,00310


In [4]:
# 3. Load in unfiltered dataset

zvhi = pd.read_excel("../Data/Ca_Zip_Home_Value.xlsx", header=1)
zvhi.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,1/31/2000,...,4/30/2025,5/31/2025,6/30/2025,7/31/2025,8/31/2025,9/30/2025,10/31/2025,11/30/2025,12/31/2025,1/31/2026
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,204988.3686,...,486136.1090,484000.3527,481473.2672,479455.2354,478365.6216,478562.4520,479293.4729,480341.6326,480930.8672,480543.6336
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,111719.9414,...,529389.7484,532145.3507,534655.0414,536426.4444,537661.4257,540194.5829,545049.8352,550925.6822,557122.8387,561559.0286
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,102728.1881,...,274073.7545,273277.6176,272287.6011,271356.0568,270430.1504,269659.0928,268926.6435,268297.3865,267961.7208,267417.8897
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,172864.4900,...,528122.2293,528512.4396,530570.0400,533309.6749,534970.6966,536204.0719,537227.4679,539139.3570,541823.1780,546584.2974
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,102246.1487,...,270027.6193,269137.4526,268139.3696,267230.2256,266357.8020,265483.6547,264636.7740,263910.1904,263568.2033,263066.0976


In [5]:
# 4. Filter ZVHI to CA + (San Diego County OR Los Angeles County) and extract unique city names

zvhi_ca = zvhi[zvhi["State"] == "CA"].copy()

zvhi_sd_la = zvhi_ca[zvhi_ca["CountyName"].isin(["San Diego County", "Los Angeles County"])].copy()

zvhi_city_names = sorted(zvhi_sd_la["City"].dropna().unique())

len(zvhi_city_names), zvhi_city_names[:25]

(135,
 ['Acton',
  'Agoura Hills',
  'Alhambra',
  'Alpine',
  'Altadena',
  'Arcadia',
  'Artesia',
  'Avalon',
  'Azusa',
  'Baldwin Park',
  'Bell',
  'Bellflower',
  'Beverly Hills',
  'Bonita',
  'Bonsall',
  'Boulevard',
  'Bradbury',
  'Burbank',
  'Calabasas',
  'Campo',
  'Carlsbad',
  'Carson',
  'Castaic',
  'Cerritos',
  'Chula Vista'])

In [6]:
# 5. Create ACS-style city name format from Zillow city list
    # This makes the ZHVI cities match the ACS city format

zvhi_city_names_acs = [
    f"{city} city, California"
    for city in zvhi_city_names
]

zvhi_city_names_acs[:20]

['Acton city, California',
 'Agoura Hills city, California',
 'Alhambra city, California',
 'Alpine city, California',
 'Altadena city, California',
 'Arcadia city, California',
 'Artesia city, California',
 'Avalon city, California',
 'Azusa city, California',
 'Baldwin Park city, California',
 'Bell city, California',
 'Bellflower city, California',
 'Beverly Hills city, California',
 'Bonita city, California',
 'Bonsall city, California',
 'Boulevard city, California',
 'Bradbury city, California',
 'Burbank city, California',
 'Calabasas city, California',
 'Campo city, California']

In [7]:
# 6. Check how many Zillow cities match ACS places, should be 135
    # 94/135 match

matches = ca_places[ca_places["NAME"].isin(zvhi_city_names_acs)]

len(matches), matches.head()

(94,
                              NAME  B01003_001E state  place
 5   Agoura Hills city, California        19721    06  00394
 15      Alhambra city, California        81022    06  00884
 49       Arcadia city, California        55170    06  02462
 56       Artesia city, California        15889    06  02896
 68        Avalon city, California         3338    06  03274)

In [8]:
# 6a. Single step: report unmatched cities from the simple "City city, California" mapping
    # this shows 30/41 missing cities are CDP (unincorporated)
acs_names = set(ca_places["NAME"].astype(str))
zvhi_raw = sorted(zvhi_city_names)                            # raw city names from Zillow
zvhi_acs = [f"{c} city, California" for c in zvhi_raw]       # current ACS-style list

unmatched = [city for city, acs in zip(zvhi_raw, zvhi_acs) if acs not in acs_names]

print("total_zillow_cities =", len(zvhi_raw))
print("matched_with_simple_city_suffix =", len(zvhi_raw) - len(unmatched))
print("unmatched_count =", len(unmatched))
print("\nFirst 30 unmatched (sample):")
print(unmatched[:30])

total_zillow_cities = 135
matched_with_simple_city_suffix = 94
unmatched_count = 41

First 30 unmatched (sample):
['Acton', 'Alpine', 'Altadena', 'Bonita', 'Bonsall', 'Boulevard', 'Campo', 'Castaic', 'Descanso', 'Dulzura', 'East Los Angeles', 'Fallbrook', 'Florence-Graham', 'Hacienda Heights', 'Jacumba', 'Jamul', 'Julian', 'La Canada Flintridge', 'La Crescenta-Montrose', 'Ladera Heights', 'Lake Hughes', 'Lakeside', 'Littlerock', 'Llano', 'Marina del Rey', 'Pala', 'Palomar Mountain', 'Pauma Valley', 'Pearblossom', 'Ramona']


In [9]:
# 6b. Check which unmatched cities exist as CDPs in ACS
    # prints the 30 CDP's
cdp_matches = []

for city in unmatched:
    candidate = f"{city} CDP, California"
    if candidate in acs_names:
        cdp_matches.append(candidate)

len(cdp_matches), cdp_matches[:20]

(30,
 ['Acton CDP, California',
  'Alpine CDP, California',
  'Altadena CDP, California',
  'Bonita CDP, California',
  'Bonsall CDP, California',
  'Boulevard CDP, California',
  'Campo CDP, California',
  'Castaic CDP, California',
  'Descanso CDP, California',
  'East Los Angeles CDP, California',
  'Fallbrook CDP, California',
  'Florence-Graham CDP, California',
  'Hacienda Heights CDP, California',
  'Jacumba CDP, California',
  'Jamul CDP, California',
  'Julian CDP, California',
  'La Crescenta-Montrose CDP, California',
  'Ladera Heights CDP, California',
  'Lake Hughes CDP, California',
  'Littlerock CDP, California'])

In [10]:
# 6c. Remove CDP-resolved cities from unmatched
    # prints the 11 nonCDP's
resolved_cdp_cities = [name.replace(" CDP, California", "") for name in cdp_matches]

still_unmatched = [city for city in unmatched if city not in resolved_cdp_cities]

len(still_unmatched), still_unmatched

(11,
 ['Dulzura',
  'La Canada Flintridge',
  'Lakeside',
  'Llano',
  'Palomar Mountain',
  'Pauma Valley',
  'Pearblossom',
  'Ranchita',
  'Santa Ysabel',
  'Spring Valley',
  'Warner Springs'])

In [11]:
# 6d. Single step: normalized-match the remaining unmatched cities against ACS names
    # normalization
import re

def normalize(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\b(city|cdp|town|village)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

acs_norm_map = {}
for a in acs_names:
    acs_norm_map.setdefault(normalize(a), []).append(a)

still_unmatched = ['Dulzura','La Canada Flintridge','Lakeside','Llano',
                   'Palomar Mountain','Pauma Valley','Pearblossom',
                   'Ranchita','Santa Ysabel','Spring Valley','Warner Springs']

results = {}
for city in still_unmatched:
    n = normalize(city)
    results[city] = acs_norm_map.get(n, [])

results

{'Dulzura': [],
 'La Canada Flintridge': [],
 'Lakeside': [],
 'Llano': [],
 'Palomar Mountain': [],
 'Pauma Valley': [],
 'Pearblossom': [],
 'Ranchita': [],
 'Santa Ysabel': [],
 'Spring Valley': [],
 'Warner Springs': []}

In [12]:
# 6e. Final ACS-matched Zillow cities (city + CDP)
final_acs_names = set(matches["NAME"]).union(set(cdp_matches))

len(final_acs_names)

124

In [13]:
# 7. Final ACS subset aligned with Zillow cities
    # 124, 4
        # This drops the 11 non census cities to make the list 124 cities

final_acs_df = ca_places[ca_places["NAME"].isin(final_acs_names)].copy()

final_acs_df.shape

(124, 4)

In [14]:
# 8. Create GEOID for consistent merging across years
    # creates clean geographic ID that's reusable across all years

final_acs_df["GEOID"] = final_acs_df["state"] + final_acs_df["place"]

final_acs_df = final_acs_df[["GEOID", "NAME", "B01003_001E"]]

final_acs_df.head()

,GEOID,NAME,B01003_001E
2,0600212,"Acton CDP, California",7880
5,0600394,"Agoura Hills city, California",19721
15,0600884,"Alhambra city, California",81022
24,0601192,"Alpine CDP, California",15180
27,0601290,"Altadena CDP, California",41705


In [15]:
# 9. Define ACS 5-year panel years

years = list(range(2009, 2025))
years

[2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024]

In [16]:
# 9a. Pull ACS total population (B01003_001E) for 2009-2024 into a panel keyed by GEOID
    # Merge w 2009-2024 Population numbers
import time

years = list(range(2009, 2025))
vars_to_get = ["NAME", "B01003_001E"]   # name + total population
panel_df = final_acs_df[["GEOID", "NAME"]].set_index("GEOID").copy()

for yr in years:
    BASE = f"https://api.census.gov/data/{yr}/acs/acs5"
    params = {
        "get": ",".join(vars_to_get),
        "for": "place:*",
        "in": f"state:{STATE_FIPS_CA}",
        "key": API_KEY
    }
    print(f"Fetching {yr}...", end=" ", flush=True)
    r = requests.get(BASE, params=params)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    # create GEOID and keep only places we care about
    df["GEOID"] = df["state"] + df["place"]
    df["B01003_001E"] = pd.to_numeric(df["B01003_001E"], errors="coerce")
    df_sub = df.set_index("GEOID")[["NAME", "B01003_001E"]].reindex(panel_df.index)
    panel_df[f"pop_{yr}"] = df_sub["B01003_001E"]
    print("done")
    time.sleep(1)  # polite pause to avoid hammering the API

# quick checks
panel_df.reset_index(inplace=True)
print("\npanel shape:", panel_df.shape)
panel_df.head()

Fetching 2009... done
Fetching 2010... done
Fetching 2011... done
Fetching 2012... done
Fetching 2013... done
Fetching 2014... done
Fetching 2015... done
Fetching 2016... done
Fetching 2017... done
Fetching 2018... done
Fetching 2019... done
Fetching 2020... done
Fetching 2021... done
Fetching 2022... done
Fetching 2023... done
Fetching 2024... done

panel shape: (124, 18)


,GEOID,NAME,pop_2009,pop_2010,pop_2011,pop_2012,pop_2013,pop_2014,pop_2015,pop_2016,pop_2017,pop_2018,pop_2019,pop_2020,pop_2021,pop_2022,pop_2023,pop_2024
0,0600212,"Acton CDP, California",1804.0,7150.0,7138.0,7117.0,7022.0,6956.0,7058,7170,7278,7185,7232,7054,6809,7286,7721,7880
1,0600394,"Agoura Hills city, California",22207.0,20328.0,20353.0,20431.0,20488.0,20630.0,20697,20689,20736,20636,20533,20269,20362,20088,19881,19721
2,0600884,"Alhambra city, California",84959.0,83389.0,83301.0,83513.0,83799.0,84400.0,84782,84866,85168,84974,84647,84509,83174,82295,81483,81022
3,0601192,"Alpine CDP, California",12627.0,13326.0,13332.0,13511.0,13717.0,14580.0,14830,15188,14795,15233,14878,14958,15648,15911,15470,15180
4,0601290,"Altadena CDP, California",44693.0,45446.0,45006.0,44578.0,45015.0,44622.0,44621,45001,45236,45146,44850,43037,43384,42719,41921,41705


In [17]:
# 10. Convert population panel to long format

panel_long = panel_df.melt(
    id_vars=["GEOID", "NAME"],
    var_name="year",
    value_name="population"
)

# Clean year column
panel_long["year"] = panel_long["year"].str.replace("pop_", "").astype(int)

panel_long.head()

,GEOID,NAME,year,population
0,0600212,"Acton CDP, California",2009,1804.0
1,0600394,"Agoura Hills city, California",2009,22207.0
2,0600884,"Alhambra city, California",2009,84959.0
3,0601192,"Alpine CDP, California",2009,12627.0
4,0601290,"Altadena CDP, California",2009,44693.0


In [18]:
# 11. Create mapping from normalized city name to GEOID
    # confirm the city + unique GEOID works
def normalize_city(s):
    return s.lower().replace(" city, california", "").replace(" cdp, california", "").strip()

acs_lookup = {
    normalize_city(name): geoid
    for geoid, name in zip(panel_df["GEOID"], panel_df["NAME"])
}

list(acs_lookup.items())[:10]

[('acton', '0600212'),
 ('agoura hills', '0600394'),
 ('alhambra', '0600884'),
 ('alpine', '0601192'),
 ('altadena', '0601290'),
 ('arcadia', '0602462'),
 ('artesia', '0602896'),
 ('avalon', '0603274'),
 ('azusa', '0603386'),
 ('baldwin park', '0603666')]

In [19]:
# 12. Attach ACS GEOID to each Zillow row using Zillow City
    # Attach unique GEOID to each Zillow city

zvhi_sd_la["city_norm"] = zvhi_sd_la["City"].astype(str).str.lower().str.strip()
zvhi_sd_la["GEOID"] = zvhi_sd_la["city_norm"].map(acs_lookup)

# Check match rate
total = len(zvhi_sd_la)
matched = zvhi_sd_la["GEOID"].notna().sum()
unmatched = total - matched

total, matched, unmatched

(369, np.int64(357), np.int64(12))

In [20]:
# 12a. Show Zillow city names that did not map to an ACS GEOID
sorted(
    zvhi_sd_la.loc[
        zvhi_sd_la["GEOID"].isna(),
        "City"
    ].unique()
)

['Dulzura',
 'La Canada Flintridge',
 'Lakeside',
 'Llano',
 'Palomar Mountain',
 'Pauma Valley',
 'Pearblossom',
 'Ranchita',
 'Santa Ysabel',
 'Spring Valley',
 'Warner Springs']

In [21]:
# 13. Remove ZIP rows that do not map to ACS places
    # dropped these 11 cities from the df

zvhi_aligned = zvhi_sd_la[zvhi_sd_la["GEOID"].notna()].copy()

zvhi_aligned.shape

(357, 324)

In [22]:
# 14. Raw ACS columns required to construct the 18 modeled variables

acs_vars = [

    # --------------------------------------------------
    # 1. Median household income
    # --------------------------------------------------
    "B19013_001E",  # Median household income (USD)

    # --------------------------------------------------
    # 2. Per capita income
    # --------------------------------------------------
    "B19301_001E",  # Per capita income (USD)

    # --------------------------------------------------
    # 3. Poverty rate (derived)
    # poverty_rate = below_poverty / poverty_universe
    # --------------------------------------------------
    "B17001_001E",  # Total population for whom poverty status is determined
    "B17001_002E",  # Population below poverty line

    # --------------------------------------------------
    # 4. Unemployment rate (derived)
    # unemployment_rate = unemployed / labor_force
    # --------------------------------------------------
    "B23025_003E",  # Civilian labor force
    "B23025_005E",  # Unemployed population

    # --------------------------------------------------
    # 5. Total population
    # --------------------------------------------------
    "B01003_001E",  # Total population

    # --------------------------------------------------
    # 7. Median age
    # --------------------------------------------------
    "B01002_001E",  # Median age (years)

    # --------------------------------------------------
    # 8. % Bachelor’s degree or higher (derived)
    # bachelors_or_higher = sum(bachelor+ degrees) / total 25+
    # --------------------------------------------------
    "B15003_001E",  # Total population age 25+
    "B15003_022E",  # Bachelor's degree
    "B15003_023E",  # Master's degree
    "B15003_024E",  # Professional degree
    "B15003_025E",  # Doctorate degree

    # --------------------------------------------------
    # 10. Total housing units
    # --------------------------------------------------
    "B25001_001E",  # Total housing units

    # --------------------------------------------------
    # 12. Vacancy rate (derived)
    # vacancy_rate = vacant_units / total_units
    # --------------------------------------------------
    "B25002_001E",  # Total housing units
    "B25002_003E",  # Vacant housing units

    # --------------------------------------------------
    # 13. Owner-occupied rate (derived)
    # owner_rate = owner_occupied / total_occupied
    # --------------------------------------------------
    "B25003_001E",  # Total occupied housing units
    "B25003_002E",  # Owner-occupied units

    # --------------------------------------------------
    # 14. % Built after 2000 (derived)
    # built_after_2000 = units_2000+ / total_units
    # --------------------------------------------------
    "B25034_001E",  # Total housing units (by year built)
    "B25034_010E",  # Built 2000–2009
    "B25034_011E",  # Built 2010 or later

    # --------------------------------------------------
    # 15. % Single-family detached (derived)
    # single_family_rate = detached_units / total_units
    # --------------------------------------------------
    "B25024_001E",  # Total housing units (by structure type)
    "B25024_002E",  # 1-unit detached structure

    # --------------------------------------------------
    # 16. Median home value
    # --------------------------------------------------
    "B25077_001E",  # Median home value (USD)

    # --------------------------------------------------
    # 17. Median gross rent
    # --------------------------------------------------
    "B25064_001E",  # Median gross rent (USD)

    # --------------------------------------------------
    # 18. % Owners cost-burdened (derived)
    # cost_burden_rate = owners paying >30% income on housing / total owners
    # --------------------------------------------------
    "B25091_001E",  # Total owner-occupied units with mortgage
    "B25091_010E",  # Owners paying 30–34.9%
    "B25091_011E",  # Owners paying 35–39.9%
    "B25091_012E"   # Owners paying 40%+
]

len(acs_vars)

29

In [23]:
# 15. Pull ACS 5-year place-level data for 2009–2022,
#     automatically requesting only the variables that exist in each year.
#     Missing vars for early years will be created as NaN so columns align.

years = list(range(2009, 2023))  # 2009–2022

# ---------------------------------------------
# Helper 1: pick a working ACS5 endpoint for a year
# ---------------------------------------------
def get_acs5_base_url(year: int) -> str:
    candidates = [
        f"https://api.census.gov/data/{year}/acs/acs5",
        f"https://api.census.gov/data/{year}/acs5",
    ]
    for url in candidates:
        test = requests.get(url, params={"get": "NAME", "for": "place:*", "in": "state:06"})
        if test.status_code == 200:
            return url
    raise ValueError(f"No working ACS5 endpoint found for year {year}")

# ---------------------------------------------
# Helper 2: load that year's variable dictionary
# (this tells us what variables are valid for that year)
# ---------------------------------------------
def get_year_variable_set(year: int, base_url: str) -> set:
    vars_url = base_url + "/variables.json"
    r = requests.get(vars_url)
    if r.status_code != 200:
        raise ValueError(f"Could not fetch variables.json for {year} from {vars_url}")
    j = r.json()
    return set(j["variables"].keys())

acs_all_years = []

for year in years:
    print(f"Pulling {year}...")

    base_url = get_acs5_base_url(year)

    # Identify which of our desired raw vars actually exist for this year
    var_set = get_year_variable_set(year, base_url)
    available_vars = [v for v in acs_vars if v in var_set]
    missing_vars = [v for v in acs_vars if v not in var_set]

    # Show a short warning if anything is missing (expected in early years)
    if missing_vars:
        print(f"  {year}: missing {len(missing_vars)} vars (will be NaN). Example: {missing_vars[:3]}")

    # Request ONLY the vars that exist for this year
    params = {
        "get": ",".join(["NAME"] + available_vars),
        "for": "place:*",
        "in": "state:06",
        "key": API_KEY
    }

    response = requests.get(base_url, params=params)
    if response.status_code != 200:
        print(f"  Error {response.status_code} for {year} using {base_url}")
        print(" ", response.text[:300])
        continue  # skip this year, try the next

    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])

    # Convert pulled ACS numeric columns to numeric
    for col in available_vars:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Create missing columns (so every year has the same set of raw vars)
    for col in missing_vars:
        df[col] = pd.NA

    # Add year
    df["year"] = year

    # Reorder columns consistently
    df = df[["NAME"] + acs_vars + ["state", "place", "year"]]

    acs_all_years.append(df)

# Combine all years we successfully pulled
acs_raw_panel = pd.concat(acs_all_years, ignore_index=True)

acs_raw_panel.shape

Pulling 2009...
  2009: missing 8 vars (will be NaN). Example: ['B23025_003E', 'B23025_005E', 'B15003_001E']
Pulling 2010...
  2010: missing 8 vars (will be NaN). Example: ['B23025_003E', 'B23025_005E', 'B15003_001E']
Pulling 2011...
  2011: missing 6 vars (will be NaN). Example: ['B15003_001E', 'B15003_022E', 'B15003_023E']
Pulling 2012...
  2012: missing 1 vars (will be NaN). Example: ['B25034_011E']
Pulling 2013...
  2013: missing 1 vars (will be NaN). Example: ['B25034_011E']
Pulling 2014...
  2014: missing 1 vars (will be NaN). Example: ['B25034_011E']
Pulling 2015...
Pulling 2016...
Pulling 2017...
Pulling 2018...
Pulling 2019...
Pulling 2020...
Pulling 2021...
Pulling 2022...


(21103, 33)

In [24]:
# 16. Construct the 18 modeled variables from the raw ACS panel
#     - Computes rates safely (avoids divide-by-zero)
#     - Renames columns to readable names
#     - Adds population_growth and housing_unit_growth using year-over-year change

import numpy as np

acs = acs_raw_panel.copy()

# -----------------------------
# Helper: safe divide (returns NaN if denom is 0/NaN)
# -----------------------------
def safe_div(numer, denom):
    numer = pd.to_numeric(numer, errors="coerce")
    denom = pd.to_numeric(denom, errors="coerce")
    return np.where((denom == 0) | (pd.isna(denom)), np.nan, numer / denom)

# -----------------------------
# 1–5, 7, 10, 16–17: direct pulls (rename)
# -----------------------------
acs["median_household_income"] = acs["B19013_001E"]   # 1
acs["per_capita_income"]       = acs["B19301_001E"]   # 2
acs["total_population"]        = acs["B01003_001E"]   # 5
acs["median_age"]              = acs["B01002_001E"]   # 7
acs["total_housing_units"]     = acs["B25001_001E"]   # 10
acs["median_home_value_acs"]   = acs["B25077_001E"]   # 16
acs["median_gross_rent"]       = acs["B25064_001E"]   # 17

# -----------------------------
# 3. Poverty rate
# -----------------------------
acs["poverty_rate"] = safe_div(acs["B17001_002E"], acs["B17001_001E"])  # 3

# -----------------------------
# 4. Unemployment rate
# -----------------------------
acs["unemployment_rate"] = safe_div(acs["B23025_005E"], acs["B23025_003E"])  # 4

# -----------------------------
# 8. % bachelor’s degree or higher (25+)
# -----------------------------
bachelors_or_higher_num = (
    pd.to_numeric(acs["B15003_022E"], errors="coerce") +  # bachelor's
    pd.to_numeric(acs["B15003_023E"], errors="coerce") +  # master's
    pd.to_numeric(acs["B15003_024E"], errors="coerce") +  # professional
    pd.to_numeric(acs["B15003_025E"], errors="coerce")    # doctorate
)
acs["pct_bachelors_or_higher"] = safe_div(bachelors_or_higher_num, acs["B15003_001E"])  # 8

# -----------------------------
# 9. % graduate degree (25+)
# -----------------------------
grad_num = (
    pd.to_numeric(acs["B15003_023E"], errors="coerce") +  # master's
    pd.to_numeric(acs["B15003_024E"], errors="coerce") +  # professional
    pd.to_numeric(acs["B15003_025E"], errors="coerce")    # doctorate
)
acs["pct_graduate_degree"] = safe_div(grad_num, acs["B15003_001E"])  # 9

# -----------------------------
# 12. Vacancy rate
# -----------------------------
acs["vacancy_rate"] = safe_div(acs["B25002_003E"], acs["B25002_001E"])  # 12

# -----------------------------
# 13. Owner-occupied rate (share of occupied units)
# -----------------------------
acs["owner_occupied_rate"] = safe_div(acs["B25003_002E"], acs["B25003_001E"])  # 13

# -----------------------------
# 14. % built after 2000
# -----------------------------
built_after_2000_num = (
    pd.to_numeric(acs["B25034_010E"], errors="coerce") +  # 2000-2009
    pd.to_numeric(acs["B25034_011E"], errors="coerce")    # 2010+
)
acs["pct_built_after_2000"] = safe_div(built_after_2000_num, acs["B25034_001E"])  # 14

# -----------------------------
# 15. % single-family detached
# -----------------------------
acs["pct_single_family_detached"] = safe_div(acs["B25024_002E"], acs["B25024_001E"])  # 15

# -----------------------------
# 18. % owners cost-burdened (>30%)
# NOTE: Using B25091 (owners w/ mortgage). Early years may be NaN (API missing vars).
# -----------------------------
cost_burden_num = (
    pd.to_numeric(acs["B25091_010E"], errors="coerce") +
    pd.to_numeric(acs["B25091_011E"], errors="coerce") +
    pd.to_numeric(acs["B25091_012E"], errors="coerce")
)
acs["pct_owners_cost_burdened"] = safe_div(cost_burden_num, acs["B25091_001E"])  # 18

# -----------------------------
# 6. Population growth (YoY, derived from panel)
# 11. Housing unit growth (YoY, derived from panel)
# Compute within each place across years
# -----------------------------
acs = acs.sort_values(["state", "place", "year"])

acs["population_growth"] = (
    acs.groupby(["state", "place"])["total_population"]
       .pct_change()
)

acs["housing_unit_growth"] = (
    acs.groupby(["state", "place"])["total_housing_units"]
       .pct_change()
)

# -----------------------------
# Keep only the final 18 variables + identifiers
# -----------------------------
acs_features = acs[[
    "NAME", "state", "place", "year",

    # 1–4
    "median_household_income",
    "per_capita_income",
    "poverty_rate",
    "unemployment_rate",

    # 5–7
    "total_population",
    "population_growth",
    "median_age",

    # 8–9
    "pct_bachelors_or_higher",
    "pct_graduate_degree",

    # 10–15
    "total_housing_units",
    "housing_unit_growth",
    "vacancy_rate",
    "owner_occupied_rate",
    "pct_built_after_2000",
    "pct_single_family_detached",

    # 16–18
    "median_home_value_acs",
    "median_gross_rent",
    "pct_owners_cost_burdened"
]].copy()

acs_features.shape

(21103, 22)

In [25]:
# 17. Keep only ACS rows for the places (cities) that appear in zvhi_aligned via GEOID

# GEOID in your workflow is the 7-digit place GEOID stored as a string
# ACS uses state + place; GEOID = state + place (for CA, state='06')

# Build the set of GEOIDs present in your Zillow-aligned dataset
geoids_used = set(zvhi_aligned["GEOID"].dropna().astype(str))

# Create GEOID in ACS features to filter with the same key
acs_features["GEOID"] = acs_features["state"].astype(str) + acs_features["place"].astype(str)

# Filter down to only the mapped cities
acs_features_sdla = acs_features[acs_features["GEOID"].isin(geoids_used)].copy()

# Sanity checks
print("Unique GEOIDs in Zillow:", len(geoids_used))
print("Unique GEOIDs in ACS after filter:", acs_features_sdla["GEOID"].nunique())
acs_features_sdla.shape

Unique GEOIDs in Zillow: 124
Unique GEOIDs in ACS after filter: 124


(1722, 23)

In [26]:
# 18. Build monthly ZHVI panel + merge annual ACS features
#     - ZHVI stays monthly (no aggregation)
#     - We add a derived 'year' column from the monthly date
#     - Merge ACS on GEOID + year so each month inherits that year's ACS values

# ---------------------------------------------------------
# A) Identify ID columns and monthly ZHVI date columns
# ---------------------------------------------------------
id_cols = ["RegionID", "City", "CountyName", "State", "GEOID"]

# Date columns in your Zillow file are strings like "2018-01-31" (or similar)
date_cols = [c for c in zvhi_aligned.columns if isinstance(c, str) and c[:4].isdigit()]

print("ID cols found:", [c for c in id_cols if c in zvhi_aligned.columns])
print("Number of monthly ZHVI columns:", len(date_cols))
print("First 3 date cols:", date_cols[:3])
print("Last 3 date cols:", date_cols[-3:])

# ---------------------------------------------------------
# B) Convert ZHVI from wide format to long format (monthly rows)
#     wide: one row per city with many date columns
#     long: one row per city-month
# ---------------------------------------------------------
zvhi_long = zvhi_aligned[id_cols + date_cols].melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="date",
    value_name="zhvi"
)

# ---------------------------------------------------------
# C) Parse the 'date' strings into real datetimes and create 'year'
#     - This does NOT change monthly ZHVI values
#     - It only adds a merge key for annual ACS data
# ---------------------------------------------------------
zvhi_long["date"] = pd.to_datetime(zvhi_long["date"], errors="coerce")
zvhi_long = zvhi_long.dropna(subset=["date"]).copy()

zvhi_long["year"] = zvhi_long["date"].dt.year

# ---------------------------------------------------------
# D) Ensure GEOID/year types match on both datasets
#     - Prevents silent merge failures due to dtype mismatch
# ---------------------------------------------------------
zvhi_long["GEOID"] = zvhi_long["GEOID"].astype(str)
zvhi_long["year"] = zvhi_long["year"].astype(int)

acs_features_sdla["GEOID"] = acs_features_sdla["GEOID"].astype(str)
acs_features_sdla["year"] = acs_features_sdla["year"].astype(int)

# ---------------------------------------------------------
# E) Merge annual ACS features onto monthly ZHVI rows
#     - Each month in a given year receives the same ACS values for that year
# ---------------------------------------------------------
zvhi_with_acs = zvhi_long.merge(
    acs_features_sdla.drop(columns=["NAME", "state", "place"]),
    on=["GEOID", "year"],
    how="left"
)

# ---------------------------------------------------------
# F) Sanity checks: confirm row counts and missingness
# ---------------------------------------------------------
print("ZHVI long shape:", zvhi_long.shape)
print("After merge shape:", zvhi_with_acs.shape)

# Pick one ACS feature as a match indicator
match_indicator = "median_household_income"
if match_indicator in zvhi_with_acs.columns:
    print("ACS missing rate (median_household_income):",
          zvhi_with_acs[match_indicator].isna().mean())
else:
    print(f"Warning: '{match_indicator}' not found in merged df")

zvhi_with_acs.head()

ID cols found: ['RegionID', 'City', 'CountyName', 'State', 'GEOID']
Number of monthly ZHVI columns: 0
First 3 date cols: []
Last 3 date cols: []
ZHVI long shape: (0, 8)
After merge shape: (0, 26)
ACS missing rate (median_household_income): nan


,RegionID,City,CountyName,State,GEOID,date,zhvi,year,median_household_income,per_capita_income,...,pct_graduate_degree,total_housing_units,housing_unit_growth,vacancy_rate,owner_occupied_rate,pct_built_after_2000,pct_single_family_detached,median_home_value_acs,median_gross_rent,pct_owners_cost_burdened


In [27]:
# 18A. Inspect columns to identify the ZHVI monthly time-series column naming pattern

cols = zvhi_aligned.columns.astype(str).tolist()

print("Total columns:", len(cols))
print("First 30 columns:", cols[:30])
print("Last 30 columns:", cols[-30:])

# Show columns that contain a year-like pattern anywhere (e.g., '2018', '2020')
year_like = [c for c in cols if "20" in c]
print("Count of columns containing '20':", len(year_like))
print("Sample of columns containing '20':", year_like[:20])

Total columns: 324
First 30 columns: ['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName', 'State', 'City', 'Metro', 'CountyName', '1/31/2000', '2/29/2000', '3/31/2000', '4/30/2000', '5/31/2000', '6/30/2000', '7/31/2000', '8/31/2000', '9/30/2000', '10/31/2000', '11/30/2000', '12/31/2000', '1/31/2001', '2/28/2001', '3/31/2001', '4/30/2001', '5/31/2001', '6/30/2001', '7/31/2001', '8/31/2001', '9/30/2001']
Last 30 columns: ['10/31/2023', '11/30/2023', '12/31/2023', '1/31/2024', '2/29/2024', '3/31/2024', '4/30/2024', '5/31/2024', '6/30/2024', '7/31/2024', '8/31/2024', '9/30/2024', '10/31/2024', '11/30/2024', '12/31/2024', '1/31/2025', '2/28/2025', '3/31/2025', '4/30/2025', '5/31/2025', '6/30/2025', '7/31/2025', '8/31/2025', '9/30/2025', '10/31/2025', '11/30/2025', '12/31/2025', '1/31/2026', 'city_norm', 'GEOID']
Count of columns containing '20': 313
Sample of columns containing '20': ['1/31/2000', '2/29/2000', '3/31/2000', '4/30/2000', '5/31/2000', '6/30/2000', '7/31/2000', '8/

In [28]:
# 18B. Identify monthly ZHVI date columns in MM/DD/YYYY format

id_cols = ["RegionID", "City", "CountyName", "State", "GEOID"]

# Date columns look like '1/31/2000' so detect by the presence of '/'
date_cols = [c for c in zvhi_aligned.columns if isinstance(c, str) and "/" in c]

print("Monthly date columns:", len(date_cols))
print("First 3:", date_cols[:3])
print("Last 3:", date_cols[-3:])

Monthly date columns: 313
First 3: ['1/31/2000', '2/29/2000', '3/31/2000']
Last 3: ['11/30/2025', '12/31/2025', '1/31/2026']


In [29]:
# 18C. Wide -> long (monthly), add year, merge ACS

# --- wide to long ---
zvhi_long = zvhi_aligned[id_cols + date_cols].melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="date",
    value_name="zhvi"
)

# --- parse date + derive year (ZHVI remains monthly) ---
zvhi_long["date"] = pd.to_datetime(zvhi_long["date"], format="%m/%d/%Y", errors="coerce")
zvhi_long = zvhi_long.dropna(subset=["date"]).copy()
zvhi_long["year"] = zvhi_long["date"].dt.year.astype(int)

# --- type alignment for merge keys ---
zvhi_long["GEOID"] = zvhi_long["GEOID"].astype(str)
acs_features_sdla["GEOID"] = acs_features_sdla["GEOID"].astype(str)

# --- merge annual ACS onto monthly ZHVI using GEOID + year ---
zvhi_with_acs = zvhi_long.merge(
    acs_features_sdla.drop(columns=["NAME", "state", "place"]),
    on=["GEOID", "year"],
    how="left"
)

# --- sanity checks ---
print("ZHVI long shape:", zvhi_long.shape)
print("After merge shape:", zvhi_with_acs.shape)
print("ACS missing rate (median_household_income):", zvhi_with_acs["median_household_income"].isna().mean())

zvhi_with_acs.head()

ZHVI long shape: (111741, 8)
After merge shape: (111741, 26)
ACS missing rate (median_household_income): 0.4651918275297339


,RegionID,City,CountyName,State,GEOID,date,zhvi,year,median_household_income,per_capita_income,...,pct_graduate_degree,total_housing_units,housing_unit_growth,vacancy_rate,owner_occupied_rate,pct_built_after_2000,pct_single_family_detached,median_home_value_acs,median_gross_rent,pct_owners_cost_burdened
0,95992,Los Angeles,Los Angeles County,CA,0644000,2000-01-31,115350.92,2000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,96193,Norwalk,Los Angeles County,CA,0652526,2000-01-31,156967.64,2000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,96361,Los Angeles,Los Angeles County,CA,0644000,2000-01-31,122178.14,2000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,96025,Los Angeles,Los Angeles County,CA,0644000,2000-01-31,131738.84,2000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,96083,Bell,Los Angeles County,CA,0604870,2000-01-31,143054.50,2000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# 19. Keep only years >= 2009 (drop 2000–2008)
#     Do NOT forward-fill yet — we need real ACS values to compute growth correctly
    # 2009-2026 ZHVI data w 73,185 rows and 26 columns

zvhi_model_panel = zvhi_with_acs[
    zvhi_with_acs["year"] >= 2009
].copy()

print("Shape after dropping 2000–2008:", zvhi_model_panel.shape)

# Confirm ACS year coverage
print("Min year:", zvhi_model_panel["year"].min())
print("Max year:", zvhi_model_panel["year"].max())

# Check missingness before synthetic projection
print("Current missing rate (median_household_income):",
      zvhi_model_panel["median_household_income"].isna().mean())

Shape after dropping 2000–2008: (73185, 26)
Min year: 2009
Max year: 2026
Current missing rate (median_household_income): 0.1834392293502767


In [31]:
# 20. Compute 5-year average growth parameters per GEOID (2017–2022)
#     Use ONLY annual ACS (acs_features_sdla), not the monthly ZHVI panel.

# ------------------------------------------------------------
# A) Choose the window of real ACS years to estimate "typical" growth
# ------------------------------------------------------------
growth_start = 2017   # first year in the growth window
growth_end   = 2022   # last year in the growth window (last real ACS year we have)

In [32]:
# 20b. Work from the annual ACS table (one row per GEOID-year)
# ------------------------------------------------------------
acs_annual = acs_features_sdla.copy()                     # make a copy so we don't modify original
acs_annual = acs_annual.sort_values(["GEOID", "year"])    # ensure rows are ordered for pct_change/diff

# Keep only the years we want for estimating growth
acs_growth_window = acs_annual[
    (acs_annual["year"] >= growth_start) &
    (acs_annual["year"] <= growth_end)
].copy()

# Quick sanity check: confirm the window is correct
print("Growth window:", acs_growth_window["year"].min(), "to", acs_growth_window["year"].max())
print("Unique GEOIDs in window:", acs_growth_window["GEOID"].nunique())

Growth window: 2017 to 2022
Unique GEOIDs in window: 124


In [33]:
# 20c. Define which variables are "levels" vs "rates"
#    Levels: project multiplicatively using % growth
#    Rates:  project additively using absolute change
# ------------------------------------------------------------
level_vars = [
    "median_household_income",   # 1
    "per_capita_income",         # 2
    "total_population",          # 5
    "median_age",                # 7
    "total_housing_units",       # 10
    "median_home_value_acs",     # 16
    "median_gross_rent",         # 17
]

rate_vars = [
    "poverty_rate",              # 3
    "unemployment_rate",         # 4
    "pct_bachelors_or_higher",   # 8
    "pct_graduate_degree",       # 9
    "vacancy_rate",              # 12
    "owner_occupied_rate",       # 13
    "pct_built_after_2000",      # 14
    "pct_single_family_detached",# 15
    "pct_owners_cost_burdened",  # 18
]

# (Optional) sanity check: confirm columns exist
missing = [c for c in level_vars + rate_vars if c not in acs_growth_window.columns]
if missing:
    raise ValueError(f"Missing expected ACS feature columns: {missing}")

In [34]:
# 20d.5 Compute yearly percent changes for level variables within each GE0ID
level_growth = (
    acs_growth_window
    .groupby("GEOID")[level_vars]
    .pct_change()
)

In [35]:
# 20d. Average the yearly % changes to get one growth rate per GE0ID
level_growth_avg = (
    pd.concat([acs_growth_window[["GEOID"]], level_growth], axis=1)
    .groupby("GEOID")
    .mean(numeric_only=True)
    .reset_index()
)

level_growth_avg = level_growth_avg.rename(columns={c: f"{c}_g" for c in level_vars})

In [36]:
# 20e. Compute yearly absolute changes for rate variables
#      Then average those changes across 2017–2022

# ---------------------------------------------
# Step 1: Compute year-over-year absolute change within each GEOID
# ---------------------------------------------
rate_change = (
    acs_growth_window
        .groupby("GEOID")[rate_vars]
        .diff()   # (current year - prior year)
)

# ---------------------------------------------
# Step 2: Average those yearly changes per GEOID
# ---------------------------------------------
rate_change_avg = (
    rate_change
        .groupby(acs_growth_window["GEOID"])
        .mean(numeric_only=True)
        .reset_index()
)

# Ensure first column is named GEOID
rate_change_avg = rate_change_avg.rename(
    columns={rate_change_avg.columns[0]: "GEOID"}
)

# Rename columns to indicate annual delta (d = delta)
rate_change_avg = rate_change_avg.rename(
    columns={c: f"{c}_d" for c in rate_vars}
)

rate_change_avg.head()

,GEOID,poverty_rate_d,unemployment_rate_d,pct_bachelors_or_higher_d,pct_graduate_degree_d,vacancy_rate_d,owner_occupied_rate_d,pct_built_after_2000_d,pct_single_family_detached_d,pct_owners_cost_burdened_d
0,0600212,0.006763,-0.018429,0.006095,-0.002542,-0.002180,0.002110,0.002684,-0.004983,-0.011764
1,0600394,-0.004791,0.002078,0.026424,0.010039,0.001297,0.008928,0.002406,0.004098,-0.002570
2,0600884,-0.005078,-0.000197,0.009629,0.004171,0.000791,0.001324,-0.005258,-0.003073,0.000345
3,0601192,0.002906,0.003412,0.002977,-0.002081,0.001335,-0.000849,0.001328,0.001685,-0.017235
4,0601290,-0.007522,0.001501,0.015538,0.008211,-0.001638,0.009862,0.003784,0.000937,0.003390


In [37]:
# 20f. Recombine it all
acs_growth_params = level_growth_avg.merge(
    rate_change_avg,
    on="GEOID",
    how="outer"
)

acs_growth_params.shape

(124, 17)

In [38]:
# 21. Create synthetic ACS annual rows for 2023–2026

projection_years = [2023, 2024, 2025, 2026]

# ------------------------------------------
# A) Extract 2022 real ACS values (anchor)
# ------------------------------------------
acs_2022 = acs_features_sdla[
    acs_features_sdla["year"] == 2022
].copy()

print("2022 anchor rows:", acs_2022.shape)

2022 anchor rows: (124, 23)


In [39]:
# 21b. Merge growth parameters onto 2022 base
# ------------------------------------------
acs_projection_base = acs_2022.merge(
    acs_growth_params,
    on="GEOID",
    how="left"
)

print("Projection base shape:", acs_projection_base.shape)

Projection base shape: (124, 39)


In [40]:
# 21c. Generate synthetic rows year-by-year
# ------------------------------------------
synthetic_rows = []

for year in projection_years:
    
    years_ahead = year - 2022
    
    df_year = acs_projection_base.copy()
    df_year["year"] = year
    
    # ---- Level variables (compound growth) ----
    for col in level_vars:
        g_col = f"{col}_g"
        df_year[col] = df_year[col] * ((1 + df_year[g_col]) ** years_ahead)
    
    # ---- Rate variables (add annual delta) ----
    for col in rate_vars:
        d_col = f"{col}_d"
        df_year[col] = df_year[col] + (df_year[d_col] * years_ahead)
    
    synthetic_rows.append(
        df_year[acs_features_sdla.columns]  # keep only original ACS feature columns
    )

In [41]:
# 21d. Combine synthetic years
# ------------------------------------------
acs_synthetic = pd.concat(synthetic_rows, ignore_index=True)

print("Synthetic ACS rows:", acs_synthetic.shape)
acs_synthetic.head()

Synthetic ACS rows: (496, 23)


,NAME,state,place,year,median_household_income,per_capita_income,poverty_rate,unemployment_rate,total_population,population_growth,...,total_housing_units,housing_unit_growth,vacancy_rate,owner_occupied_rate,pct_built_after_2000,pct_single_family_detached,median_home_value_acs,median_gross_rent,pct_owners_cost_burdened,GEOID
0,"Acton CDP, California",06,00212,2023,125798.032210,50384.920351,0.109067,-0.003151,7292.517489,0.070054,...,2679.353242,-0.013469,0.071621,0.922031,0.056190,0.885054,6.898828e+05,1737.465038,0.184858,0600212
1,"Agoura Hills city, California",06,00394,2023,167938.338149,80504.157303,0.045189,0.056367,19961.287665,-0.013456,...,7419.449583,-0.043467,0.051565,0.797778,0.017956,0.692168,1.025849e+06,3246.824579,0.186194,0600394
2,"Alhambra city, California",06,00884,2023,86425.590301,41091.312616,0.120136,0.054815,81733.390493,-0.010568,...,32272.107132,0.004985,0.063186,0.411386,0.347865,0.418061,8.182797e+05,1910.242259,0.193626,0600884
3,"Alpine CDP, California",06,01192,2023,110101.598069,51497.104377,0.091538,0.062082,16148.434815,0.016807,...,5694.643732,0.001235,0.038176,0.712365,0.049627,0.714007,8.172635e+05,2050.799166,0.116201,0601192
4,"Altadena CDP, California",06,01290,2023,132481.782862,66061.705231,0.060600,0.068271,42238.538487,-0.015328,...,15555.939658,-0.008260,0.059460,0.766633,0.612405,0.868316,1.078756e+06,2266.472444,0.220105,0601290


In [42]:
# 21e. Check 2023-2025 values are truly unique for a specified column

test_geoid = "0600212"   # replace with any GEOID you want to inspect

acs_synthetic[
    (acs_synthetic["GEOID"] == test_geoid) &
    (acs_synthetic["year"].isin([2023, 2024, 2025]))
][["GEOID", "year", "median_household_income"]].sort_values("year")

,GEOID,year,median_household_income
0,0600212,2023,125798.032210
124,0600212,2024,132882.794736
248,0600212,2025,140366.560800


In [43]:
# 22. Combine real and synthetic ACS annual tables with source tagging

# ---------------------------------------
# A) Tag real ACS data
# ---------------------------------------
acs_real = acs_features_sdla.copy()
acs_real["acs_source"] = "real"

# ---------------------------------------
# B) Tag synthetic ACS data
# ---------------------------------------
acs_synthetic_tagged = acs_synthetic.copy()
acs_synthetic_tagged["acs_source"] = "synthetic"

# ---------------------------------------
# C) Combine into one annual ACS table
# ---------------------------------------
acs_full_annual = pd.concat(
    [acs_real, acs_synthetic_tagged],
    ignore_index=True
)

# Sanity check
print("Combined annual ACS shape:", acs_full_annual.shape)
print(acs_full_annual["acs_source"].value_counts())
acs_full_annual.head()

Combined annual ACS shape: (2218, 24)
acs_source
real         1722
synthetic     496
Name: count, dtype: int64


,NAME,state,place,year,median_household_income,per_capita_income,poverty_rate,unemployment_rate,total_population,population_growth,...,housing_unit_growth,vacancy_rate,owner_occupied_rate,pct_built_after_2000,pct_single_family_detached,median_home_value_acs,median_gross_rent,pct_owners_cost_burdened,GEOID,acs_source
0,"Acton CDP, California",06,00212,2009,67336.0,32165.0,0.014412,NaN,1804.0,NaN,...,NaN,0.109317,0.921897,NaN,0.741615,524300.0,1625.0,0.452345,0600212,real
1,"Acton CDP, California",06,00212,2010,90888.0,35943.0,0.062797,NaN,7150.0,2.963415,...,2.196273,0.075787,0.906224,NaN,0.882627,561900.0,2001.0,0.326682,0600212,real
2,"Acton CDP, California",06,00212,2011,87896.0,33279.0,0.083392,0.076000,7138.0,-0.001678,...,0.017489,0.077922,0.886910,NaN,0.858671,510300.0,2001.0,0.322746,0600212,real
3,"Acton CDP, California",06,00212,2012,91888.0,36689.0,0.097441,0.093864,7117.0,-0.002942,...,0.009549,0.069996,0.907242,NaN,0.874007,464300.0,2001.0,0.347982,0600212,real
4,"Acton CDP, California",06,00212,2013,89886.0,34546.0,0.116638,0.126219,7022.0,-0.013348,...,0.017404,0.101153,0.877534,NaN,0.865377,434100.0,1862.0,0.346063,0600212,real


In [44]:
# 22a. Merge combined ACS annual table (real + synthetic) onto monthly ZHVI panel
# ------------------------------------------------------------
# A) Ensure merge keys are the same type on both tables
# ------------------------------------------------------------
acs_full_annual["GEOID"] = acs_full_annual["GEOID"].astype(str)
acs_full_annual["year"]  = acs_full_annual["year"].astype(int)

zvhi_model_panel["GEOID"] = zvhi_model_panel["GEOID"].astype(str)
zvhi_model_panel["year"]  = zvhi_model_panel["year"].astype(int)

In [45]:
# 22b. Drop existing ACS columns from zvhi_model_panel (prevents duplicates)
# ------------------------------------------------------------
acs_feature_cols = [
    "median_household_income",
    "per_capita_income",
    "poverty_rate",
    "unemployment_rate",
    "total_population",
    "population_growth",
    "median_age",
    "pct_bachelors_or_higher",
    "pct_graduate_degree",
    "total_housing_units",
    "housing_unit_growth",
    "vacancy_rate",
    "owner_occupied_rate",
    "pct_built_after_2000",
    "pct_single_family_detached",
    "median_home_value_acs",
    "median_gross_rent",
    "pct_owners_cost_burdened",
    "acs_source",
]

zvhi_model_panel_noacs = zvhi_model_panel.drop(
    columns=[c for c in acs_feature_cols if c in zvhi_model_panel.columns],
    errors="ignore"
)

In [46]:
# 22c. Merge annual ACS onto monthly panel by GEOID + year
# ------------------------------------------------------------
zvhi_final_panel = zvhi_model_panel_noacs.merge(
    acs_full_annual,
    on=["GEOID", "year"],
    how="left"
)

In [47]:
# 22d. Sanity checks: confirm synthetic years are present
# ------------------------------------------------------------
print("zvhi_final_panel shape:", zvhi_final_panel.shape)
print("acs_source counts (incl NaN):")
print(zvhi_final_panel["acs_source"].value_counts(dropna=False))

# Optional: check 2023–2025 specifically
print("\nacs_source counts for 2023–2025:")
print(zvhi_final_panel.loc[zvhi_final_panel["year"].between(2023, 2025), "acs_source"]
      .value_counts(dropna=False))

zvhi_final_panel shape: (73185, 30)
acs_source counts (incl NaN):
acs_source
real         59784
synthetic    13209
NaN            192
Name: count, dtype: int64

acs_source counts for 2023–2025:
acs_source
synthetic    12852
Name: count, dtype: int64


In [48]:
# 22e. Diagnose unmatched rows (acs_source is NaN)

unmatched = zvhi_final_panel[zvhi_final_panel["acs_source"].isna()].copy()

print("Unmatched rows:", unmatched.shape[0])

print("\nUnmatched by year:")
print(unmatched["year"].value_counts().sort_index())

print("\nUnmatched by GEOID (top 20):")
print(unmatched["GEOID"].value_counts().head(20))

# show a few examples so we can see the pattern 
unmatched[["RegionID", "City", "CountyName", "State", "GEOID", "year", "date"]].head(10)

Unmatched rows: 192

Unmatched by year:
year
2009    132
2010     12
2011     12
2012     12
2013     12
2014     12
Name: count, dtype: int64

Unmatched by GEOID (top 20):
GEOID
0655058    72
0610508    24
0618940    24
0611796    12
0674130    12
0678960    12
0639556    12
0607694    12
0637022    12
Name: count, dtype: int64


,RegionID,City,CountyName,State,GEOID,year,date
257,96398,Castaic,Los Angeles County,CA,0611796,2009,2009-01-31
302,96395,Stevenson Ranch,Los Angeles County,CA,0674130,2009,2009-01-31
338,96126,Topanga,Los Angeles County,CA,0678960,2009,2009-01-31
342,96542,Campo,San Diego County,CA,0610508,2009,2009-01-31
343,97316,Lake Hughes,Los Angeles County,CA,0639556,2009,2009-01-31
347,96569,Descanso,San Diego County,CA,0618940,2009,2009-01-31
349,96551,Descanso,San Diego County,CA,0618940,2009,2009-01-31
350,96541,Boulevard,San Diego County,CA,0607694,2009,2009-01-31
351,96615,Pala,San Diego County,CA,0655058,2009,2009-01-31
354,96557,Jacumba,San Diego County,CA,0637022,2009,2009-01-31


In [49]:
# 23. Build 2018–2025 dataset

# Build clean 2018–2025 dataset (monthly, zero ACS missing values)

panel_2018_2025 = zvhi_final_panel[
    (zvhi_final_panel["year"] >= 2018) &
    (zvhi_final_panel["year"] <= 2025)
].copy()

print("2018–2025 shape:", panel_2018_2025.shape)

acs_feature_cols = [
    "median_household_income",
    "per_capita_income",
    "poverty_rate",
    "unemployment_rate",
    "total_population",
    "population_growth",
    "median_age",
    "pct_bachelors_or_higher",
    "pct_graduate_degree",
    "total_housing_units",
    "housing_unit_growth",
    "vacancy_rate",
    "owner_occupied_rate",
    "pct_built_after_2000",
    "pct_single_family_detached",
    "median_home_value_acs",
    "median_gross_rent",
    "pct_owners_cost_burdened"
]

total_nan = panel_2018_2025[acs_feature_cols].isna().sum().sum()

print("Total ACS NaNs (2018–2025):", total_nan)

2018–2025 shape: (34272, 30)
Total ACS NaNs (2018–2025): 0


In [50]:
# 23b. How many unique zips in 2018-2025

print("Unique GEOIDs (2018–2025):", panel_2018_2025["GEOID"].nunique())

Unique GEOIDs (2018–2025): 124


In [51]:
# 24. what 2009 columns are na
# Filter 2009 rows only
panel_2009 = zvhi_final_panel[zvhi_final_panel["year"] == 2009].copy()

# Count NaNs per ACS feature column
na_by_column_2009 = (
    panel_2009[acs_feature_cols]
        .isna()
        .sum()
        .sort_values(ascending=False)
)

# Show only columns that actually have missing values
na_by_column_2009 = na_by_column_2009[na_by_column_2009 > 0]

na_by_column_2009

unemployment_rate             4284
population_growth             4284
pct_bachelors_or_higher       4284
pct_graduate_degree           4284
pct_built_after_2000          4284
housing_unit_growth           4284
median_household_income        132
owner_occupied_rate            132
median_gross_rent              132
median_home_value_acs          132
pct_single_family_detached     132
total_housing_units            132
vacancy_rate                   132
per_capita_income              132
median_age                     132
total_population               132
poverty_rate                   132
pct_owners_cost_burdened       132
dtype: int64

In [52]:
# 24a. what 2010 columns are na
# Filter 2010 rows only
panel_2010 = zvhi_final_panel[zvhi_final_panel["year"] == 2010].copy()

print("Total rows in 2010:", len(panel_2010))

# Count NaNs per ACS feature column
na_by_column_2010 = (
    panel_2010[acs_feature_cols]
        .isna()
        .sum()
        .sort_values(ascending=False)
)

# Show only columns that actually have missing values
na_by_column_2010 = na_by_column_2010[na_by_column_2010 > 0]

na_by_column_2010

Total rows in 2010: 4284


unemployment_rate             4284
pct_bachelors_or_higher       4284
pct_graduate_degree           4284
pct_built_after_2000          4284
population_growth              132
housing_unit_growth            132
median_household_income         12
owner_occupied_rate             12
median_gross_rent               12
median_home_value_acs           12
pct_single_family_detached      12
total_housing_units             12
vacancy_rate                    12
per_capita_income               12
median_age                      12
total_population                12
poverty_rate                    12
pct_owners_cost_burdened        12
dtype: int64

In [53]:
# 25a. what columns are in panel_2018_2025
# Check if both ZHVI and ACS variables exist in 2018–2025 panel

print("Columns in panel_2018_2025:")
print(panel_2018_2025.columns.tolist())

Columns in panel_2018_2025:
['RegionID', 'City', 'CountyName', 'State', 'GEOID', 'date', 'zhvi', 'year', 'NAME', 'state', 'place', 'median_household_income', 'per_capita_income', 'poverty_rate', 'unemployment_rate', 'total_population', 'population_growth', 'median_age', 'pct_bachelors_or_higher', 'pct_graduate_degree', 'total_housing_units', 'housing_unit_growth', 'vacancy_rate', 'owner_occupied_rate', 'pct_built_after_2000', 'pct_single_family_detached', 'median_home_value_acs', 'median_gross_rent', 'pct_owners_cost_burdened', 'acs_source']


In [54]:
# 25b. confirm ZHVI is still monthly (it is)
print("Unique months in 2018–2025:",
      panel_2018_2025["date"].nunique())

print("Rows per year (should be 12 × ZIP count):")
print(panel_2018_2025.groupby("year").size())

Unique months in 2018–2025: 96
Rows per year (should be 12 × ZIP count):
year
2018    4284
2019    4284
2020    4284
2021    4284
2022    4284
2023    4284
2024    4284
2025    4284
dtype: int64


In [55]:
# 25c. Print first 20 rows of panel_2018_2025
# Display first 20 rows with all columns visible

# Temporarily show all columns
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

panel_2018_2025.head(20)

,RegionID,City,CountyName,State,GEOID,date,zhvi,year,NAME,state,place,median_household_income,per_capita_income,poverty_rate,unemployment_rate,total_population,population_growth,median_age,pct_bachelors_or_higher,pct_graduate_degree,total_housing_units,housing_unit_growth,vacancy_rate,owner_occupied_rate,pct_built_after_2000,pct_single_family_detached,median_home_value_acs,median_gross_rent,pct_owners_cost_burdened,acs_source
38556,95992,Los Angeles,Los Angeles County,CA,0644000,2018-01-31,340420.24,2018,"Los Angeles city, California",06,44000,58385.0,33420.0,0.190545,0.070501,3959657.0,0.002502,35.4,0.337062,0.116290,1474043.0,0.011168,0.067962,0.368388,0.305260,0.382101,599700.0,1376.0,0.237359,real
38557,96193,Norwalk,Los Angeles County,CA,0652526,2018-01-31,447875.54,2018,"Norwalk city, California",06,52526,66453.0,22076.0,0.128377,0.063058,105886.0,-0.004868,35.4,0.171741,0.041744,28475.0,-0.004197,0.045478,0.638116,0.145040,0.737981,410400.0,1488.0,0.209756,real
38558,96361,Los Angeles,Los Angeles County,CA,0644000,2018-01-31,405542.60,2018,"Los Angeles city, California",06,44000,58385.0,33420.0,0.190545,0.070501,3959657.0,0.002502,35.4,0.337062,0.116290,1474043.0,0.011168,0.067962,0.368388,0.305260,0.382101,599700.0,1376.0,0.237359,real
38559,96025,Los Angeles,Los Angeles County,CA,0644000,2018-01-31,374108.14,2018,"Los Angeles city, California",06,44000,58385.0,33420.0,0.190545,0.070501,3959657.0,0.002502,35.4,0.337062,0.116290,1474043.0,0.011168,0.067962,0.368388,0.305260,0.382101,599700.0,1376.0,0.237359,real
38560,96083,Bell,Los Angeles County,CA,0604870,2018-01-31,385547.57,2018,"Bell city, California",06,04870,42548.0,14455.0,0.238912,0.068535,35809.0,-0.003700,30.5,0.071603,0.006232,9095.0,-0.015266,0.029687,0.287365,0.363386,0.448708,390100.0,1137.0,0.230284,real
38561,96107,Hawthorne,Los Angeles County,CA,0632548,2018-01-31,545797.62,2018,"Hawthorne city, California",06,32548,50948.0,23792.0,0.155534,0.073607,87370.0,-0.000629,33.4,0.215775,0.061299,30656.0,-0.010714,0.044853,0.263174,0.144768,0.289960,499200.0,1188.0,0.229042,real
38562,96239,Long Beach,Los Angeles County,CA,0643000,2018-01-31,403077.82,2018,"Long Beach city, California",06,43000,60551.0,31114.0,0.181418,0.070864,468883.0,-0.003413,34.6,0.305795,0.109119,175235.0,0.008599,0.050076,0.400691,0.335361,0.399190,519300.0,1252.0,0.193556,real
38563,96125,South Gate,Los Angeles County,CA,0673080,2018-01-31,408275.38,2018,"South Gate city, California",06,73080,50246.0,17174.0,0.187708,0.103635,95103.0,-0.003322,31.6,0.093034,0.025793,24385.0,0.010191,0.018823,0.421968,0.480008,0.554767,375900.0,1125.0,0.248613,real
38564,96368,Los Angeles,Los Angeles County,CA,0644000,2018-01-31,452364.57,2018,"Los Angeles city, California",06,44000,58385.0,33420.0,0.190545,0.070501,3959657.0,0.002502,35.4,0.337062,0.116290,1474043.0,0.011168,0.067962,0.368388,0.305260,0.382101,599700.0,1376.0,0.237359,real
38565,96546,Chula Vista,San Diego County,CA,0613392,2018-01-31,451680.23,2018,"Chula Vista city, California",06,13392,76354.0,29143.0,0.108483,0.090424,266468.0,0.008962,34.9,0.287515,0.088477,85810.0,0.004495,0.080061,0.583481,0.040461,0.533935,465000.0,1536.0,0.187408,real


In [56]:
# 25d. panel_2015_2018 columns (no RegionName)
panel_2018_2025.columns

Index(['RegionID', 'City', 'CountyName', 'State', 'GEOID', 'date', 'zhvi',
       'year', 'NAME', 'state', 'place', 'median_household_income',
       'per_capita_income', 'poverty_rate', 'unemployment_rate',
       'total_population', 'population_growth', 'median_age',
       'pct_bachelors_or_higher', 'pct_graduate_degree', 'total_housing_units',
       'housing_unit_growth', 'vacancy_rate', 'owner_occupied_rate',
       'pct_built_after_2000', 'pct_single_family_detached',
       'median_home_value_acs', 'median_gross_rent',
       'pct_owners_cost_burdened', 'acs_source'],
      dtype='object')

In [57]:
# 25e. Check if RegionName column exists in the original ZHVI dataframe

print("RegionName exists in zhvi:", "RegionName" in zvhi.columns)

# If it exists, show first few values
if "RegionName" in zvhi.columns:
    print(zvhi["RegionName"].head())

RegionName exists in zhvi: True
0    77494
1     8701
2    77449
3    11368
4    77084
Name: RegionName, dtype: int64


In [58]:
# 25f. Check if RegionName column exists in ZHVI_aligned
"RegionName" in zvhi_aligned.columns

True

In [59]:
#25g. Check firtst 2 rows of ZHVI_aligned
zvhi_aligned[["RegionID", "RegionName", "City", "State"]].head(20)

,RegionID,RegionName,City,State
9,95992,90011,Los Angeles,CA
12,96193,90650,Norwalk,CA
13,96361,91331,Los Angeles,CA
21,96025,90044,Los Angeles,CA
26,96083,90201,Bell,CA
27,96107,90250,Hawthorne,CA
30,96239,90805,Long Beach,CA
34,96125,90280,South Gate,CA
36,96368,91342,Los Angeles,CA
50,96546,91911,Chula Vista,CA


In [60]:
# 26. Build cleamn Region to Zip Identifier
# RegionID -> ZIP lookup (RegionName)
zip_lookup = (
    zvhi_aligned[["RegionID", "RegionName"]]
    .dropna(subset=["RegionID", "RegionName"])
    .drop_duplicates(subset=["RegionID"])
    .copy()
)

# Ensure ZIP is a 5-char string (keeps leading zeros)
zip_lookup["RegionName"] = zip_lookup["RegionName"].astype(str).str.zfill(5)

print("Unique RegionIDs in lookup:", zip_lookup["RegionID"].nunique())
zip_lookup.head()

Unique RegionIDs in lookup: 357


,RegionID,RegionName
9,95992,90011
12,96193,90650
13,96361,91331
21,96025,90044
26,96083,90201


In [61]:
# 26a. Attach zip_lookup to panel_2018_2025
panel_2018_2025 = panel_2018_2025.merge(zip_lookup, on="RegionID", how="left")

# Resolve duplicate ZIP column names created by merge
if "RegionName_x" in panel_2018_2025.columns and "RegionName_y" in panel_2018_2025.columns:
    panel_2018_2025 = panel_2018_2025.drop(columns=["RegionName_y"])
    panel_2018_2025 = panel_2018_2025.rename(columns={"RegionName_x": "zip_code"})
elif "RegionName" in panel_2018_2025.columns:
    panel_2018_2025 = panel_2018_2025.rename(columns={"RegionName": "zip_code"})

# Remove accidental duplicate zip_code columns if they exist
panel_2018_2025 = panel_2018_2025.loc[:, ~panel_2018_2025.columns.duplicated()]

# Sanity check
print("ZIP missing rate:", panel_2018_2025["zip_code"].isna().mean())
panel_2018_2025[["RegionID", "zip_code", "City", "CountyName", "State"]].head(10)

ZIP missing rate: 0.0


,RegionID,zip_code,City,CountyName,State
0,95992,90011,Los Angeles,Los Angeles County,CA
1,96193,90650,Norwalk,Los Angeles County,CA
2,96361,91331,Los Angeles,Los Angeles County,CA
3,96025,90044,Los Angeles,Los Angeles County,CA
4,96083,90201,Bell,Los Angeles County,CA
5,96107,90250,Hawthorne,Los Angeles County,CA
6,96239,90805,Long Beach,Los Angeles County,CA
7,96125,90280,South Gate,Los Angeles County,CA
8,96368,91342,Los Angeles,Los Angeles County,CA
9,96546,91911,Chula Vista,San Diego County,CA


In [62]:
# 26b. Attach zip lookup to other df's
# Examples (replace with your actual dataframe names)

zvhi_long      = zvhi_long.merge(zip_lookup, on="RegionID", how="left")
zvhi_with_acs  = zvhi_with_acs.merge(zip_lookup, on="RegionID", how="left")
zvhi_model_panel = zvhi_model_panel.merge(zip_lookup, on="RegionID", how="left")

In [63]:
# 26c. Sanity check. Print any unmatched zips (0)

missing_ids = panel_2018_2025[panel_2018_2025["zip_code"].isna()]["RegionID"].unique()
missing_ids[:20], len(missing_ids)

(array([], dtype=int64), 0)

In [75]:
# 27. Create two separate datasets:
# 1) model dataset = stays numeric for ML
# 2) display dataset = formatted for readability/export

panel_model = panel_2018_2025.copy()
panel_display = panel_2018_2025.copy()

In [77]:
# 27a. Confirm dataset is numeric:

print(panel_2018_2025.dtypes)

print(panel_2018_2025.select_dtypes(include="object").columns.tolist())

zip_code                       object
GEOID                          object
City                           object
CountyName                     object
date                           object
zhvi                           object
year                            int64
median_household_income        object
per_capita_income              object
poverty_rate                   object
unemployment_rate              object
total_population              float64
population_growth              object
median_age                    float64
pct_bachelors_or_higher        object
pct_graduate_degree            object
total_housing_units           float64
housing_unit_growth            object
vacancy_rate                   object
owner_occupied_rate            object
pct_built_after_2000           object
pct_single_family_detached     object
median_home_value_acs          object
median_gross_rent              object
pct_owners_cost_burdened       object
acs_source                     object
dtype: objec

In [78]:
# 28. Clean numeric columns:

# Columns that should be numeric
num_cols = [
    "zhvi", "median_household_income", "per_capita_income",
    "poverty_rate", "unemployment_rate", "population_growth",
    "pct_bachelors_or_higher", "pct_graduate_degree",
    "housing_unit_growth", "vacancy_rate",
    "owner_occupied_rate", "pct_built_after_2000",
    "pct_single_family_detached", "median_home_value_acs",
    "median_gross_rent", "pct_owners_cost_burdened"
]

# Remove $, commas, % and convert
for col in num_cols:
    panel_2018_2025[col] = (
        panel_2018_2025[col]
        .astype(str)
        .str.replace(r"[$,%]", "", regex=True)
        .str.replace(",", "", regex=True)
    )
    panel_2018_2025[col] = pd.to_numeric(panel_2018_2025[col], errors="coerce")

In [79]:
# 28a. Clean percent scale:

pct_cols = [
    "poverty_rate", "unemployment_rate", "population_growth",
    "pct_bachelors_or_higher", "pct_graduate_degree",
    "housing_unit_growth", "vacancy_rate",
    "owner_occupied_rate", "pct_built_after_2000",
    "pct_single_family_detached", "pct_owners_cost_burdened"
]

for col in pct_cols:
    panel_2018_2025[col] = panel_2018_2025[col] / 100

In [80]:
# 28c. Verify fix:

print(panel_2018_2025.dtypes)
print(panel_2018_2025.select_dtypes(include="object").columns.tolist())

zip_code                       object
GEOID                          object
City                           object
CountyName                     object
date                           object
zhvi                          float64
year                            int64
median_household_income       float64
per_capita_income               int64
poverty_rate                  float64
unemployment_rate             float64
total_population              float64
population_growth             float64
median_age                    float64
pct_bachelors_or_higher       float64
pct_graduate_degree           float64
total_housing_units           float64
housing_unit_growth           float64
vacancy_rate                  float64
owner_occupied_rate           float64
pct_built_after_2000          float64
pct_single_family_detached    float64
median_home_value_acs           int64
median_gross_rent             float64
pct_owners_cost_burdened      float64
acs_source                     object
dtype: objec

In [87]:
# 29. Recreate datasets:

panel_model = panel_2018_2025.copy()
panel_display = panel_2018_2025.copy()

In [89]:
# 30. Re-format Display dataset:

# 1. Reorder columns
ordered_cols = [
    "zip_code", "GEOID", "City", "CountyName", "date", "zhvi", "year",
    "median_household_income", "per_capita_income", "poverty_rate", "unemployment_rate",
    "total_population", "population_growth", "median_age",
    "pct_bachelors_or_higher", "pct_graduate_degree",
    "total_housing_units", "housing_unit_growth", "vacancy_rate",
    "owner_occupied_rate", "pct_built_after_2000",
    "pct_single_family_detached", "median_home_value_acs",
    "median_gross_rent", "pct_owners_cost_burdened", "acs_source"
]

panel_display = panel_display[ordered_cols]


# 2. Format currency columns
money_cols = [
    "zhvi", "median_household_income", "per_capita_income",
    "median_home_value_acs", "median_gross_rent"
]

for col in money_cols:
    panel_display[col] = panel_display[col].apply(lambda x: f"${x:,.0f}")


# 3. Format percentage columns
pct_cols = [
    "poverty_rate", "unemployment_rate", "population_growth",
    "pct_bachelors_or_higher", "pct_graduate_degree",
    "housing_unit_growth", "vacancy_rate",
    "owner_occupied_rate", "pct_built_after_2000",
    "pct_single_family_detached", "pct_owners_cost_burdened"
]

for col in pct_cols:
    panel_display[col] = (panel_display[col] * 100).round(2).astype(str) + "%"


# 4. Format date column
panel_display["date"] = pd.to_datetime(panel_display["date"]).dt.strftime("%-m/%-d/%Y")


# 5. Optional: round median_age for readability
panel_display["median_age"] = panel_display["median_age"].round(1)

In [90]:
print("MODEL DATASET:")
display(panel_model.head(10))

print("\nDISPLAY DATASET:")
display(panel_display.head(10))

MODEL DATASET:


,zip_code,GEOID,City,CountyName,date,zhvi,year,median_household_income,per_capita_income,poverty_rate,unemployment_rate,total_population,population_growth,median_age,pct_bachelors_or_higher,pct_graduate_degree,total_housing_units,housing_unit_growth,vacancy_rate,owner_occupied_rate,pct_built_after_2000,pct_single_family_detached,median_home_value_acs,median_gross_rent,pct_owners_cost_burdened,acs_source
0,90011,0644000,Los Angeles,Los Angeles County,1/31/2018,340420.0,2018,58385.0,33420,0.1905,0.0705,3959657.0,0.0025,35.4,0.3371,0.1163,1474043.0,0.0112,0.0680,0.3684,0.3053,0.3821,599700,1376.0,0.2374,real
1,90650,0652526,Norwalk,Los Angeles County,1/31/2018,447876.0,2018,66453.0,22076,0.1284,0.0631,105886.0,-0.0049,35.4,0.1717,0.0417,28475.0,-0.0042,0.0455,0.6381,0.1450,0.7380,410400,1488.0,0.2098,real
2,91331,0644000,Los Angeles,Los Angeles County,1/31/2018,405543.0,2018,58385.0,33420,0.1905,0.0705,3959657.0,0.0025,35.4,0.3371,0.1163,1474043.0,0.0112,0.0680,0.3684,0.3053,0.3821,599700,1376.0,0.2374,real
3,90044,0644000,Los Angeles,Los Angeles County,1/31/2018,374108.0,2018,58385.0,33420,0.1905,0.0705,3959657.0,0.0025,35.4,0.3371,0.1163,1474043.0,0.0112,0.0680,0.3684,0.3053,0.3821,599700,1376.0,0.2374,real
4,90201,0604870,Bell,Los Angeles County,1/31/2018,385548.0,2018,42548.0,14455,0.2389,0.0685,35809.0,-0.0037,30.5,0.0716,0.0062,9095.0,-0.0153,0.0297,0.2874,0.3634,0.4487,390100,1137.0,0.2303,real
5,90250,0632548,Hawthorne,Los Angeles County,1/31/2018,545798.0,2018,50948.0,23792,0.1555,0.0736,87370.0,-0.0006,33.4,0.2158,0.0613,30656.0,-0.0107,0.0449,0.2632,0.1448,0.2900,499200,1188.0,0.2290,real
6,90805,0643000,Long Beach,Los Angeles County,1/31/2018,403078.0,2018,60551.0,31114,0.1814,0.0709,468883.0,-0.0034,34.6,0.3058,0.1091,175235.0,0.0086,0.0501,0.4007,0.3354,0.3992,519300,1252.0,0.1936,real
7,90280,0673080,South Gate,Los Angeles County,1/31/2018,408275.0,2018,50246.0,17174,0.1877,0.1036,95103.0,-0.0033,31.6,0.0930,0.0258,24385.0,0.0102,0.0188,0.4220,0.4800,0.5548,375900,1125.0,0.2486,real
8,91342,0644000,Los Angeles,Los Angeles County,1/31/2018,452365.0,2018,58385.0,33420,0.1905,0.0705,3959657.0,0.0025,35.4,0.3371,0.1163,1474043.0,0.0112,0.0680,0.3684,0.3053,0.3821,599700,1376.0,0.2374,real
9,91911,0613392,Chula Vista,San Diego County,1/31/2018,451680.0,2018,76354.0,29143,0.1085,0.0904,266468.0,0.0090,34.9,0.2875,0.0885,85810.0,0.0045,0.0801,0.5835,0.0405,0.5339,465000,1536.0,0.1874,real



DISPLAY DATASET:


,zip_code,GEOID,City,CountyName,date,zhvi,year,median_household_income,per_capita_income,poverty_rate,unemployment_rate,total_population,population_growth,median_age,pct_bachelors_or_higher,pct_graduate_degree,total_housing_units,housing_unit_growth,vacancy_rate,owner_occupied_rate,pct_built_after_2000,pct_single_family_detached,median_home_value_acs,median_gross_rent,pct_owners_cost_burdened,acs_source
0,90011,0644000,Los Angeles,Los Angeles County,1/31/2018,"$340,420",2018,"$58,385","$33,420",19.05%,7.05%,3959657.0,0.25%,35.4,33.71%,11.63%,1474043.0,1.12%,6.8%,36.84%,30.53%,38.21%,"$599,700","$1,376",23.74%,real
1,90650,0652526,Norwalk,Los Angeles County,1/31/2018,"$447,876",2018,"$66,453","$22,076",12.84%,6.31%,105886.0,-0.49%,35.4,17.17%,4.17%,28475.0,-0.42%,4.55%,63.81%,14.5%,73.8%,"$410,400","$1,488",20.98%,real
2,91331,0644000,Los Angeles,Los Angeles County,1/31/2018,"$405,543",2018,"$58,385","$33,420",19.05%,7.05%,3959657.0,0.25%,35.4,33.71%,11.63%,1474043.0,1.12%,6.8%,36.84%,30.53%,38.21%,"$599,700","$1,376",23.74%,real
3,90044,0644000,Los Angeles,Los Angeles County,1/31/2018,"$374,108",2018,"$58,385","$33,420",19.05%,7.05%,3959657.0,0.25%,35.4,33.71%,11.63%,1474043.0,1.12%,6.8%,36.84%,30.53%,38.21%,"$599,700","$1,376",23.74%,real
4,90201,0604870,Bell,Los Angeles County,1/31/2018,"$385,548",2018,"$42,548","$14,455",23.89%,6.85%,35809.0,-0.37%,30.5,7.16%,0.62%,9095.0,-1.53%,2.97%,28.74%,36.34%,44.87%,"$390,100","$1,137",23.03%,real
5,90250,0632548,Hawthorne,Los Angeles County,1/31/2018,"$545,798",2018,"$50,948","$23,792",15.55%,7.36%,87370.0,-0.06%,33.4,21.58%,6.13%,30656.0,-1.07%,4.49%,26.32%,14.48%,29.0%,"$499,200","$1,188",22.9%,real
6,90805,0643000,Long Beach,Los Angeles County,1/31/2018,"$403,078",2018,"$60,551","$31,114",18.14%,7.09%,468883.0,-0.34%,34.6,30.58%,10.91%,175235.0,0.86%,5.01%,40.07%,33.54%,39.92%,"$519,300","$1,252",19.36%,real
7,90280,0673080,South Gate,Los Angeles County,1/31/2018,"$408,275",2018,"$50,246","$17,174",18.77%,10.36%,95103.0,-0.33%,31.6,9.3%,2.58%,24385.0,1.02%,1.88%,42.2%,48.0%,55.48%,"$375,900","$1,125",24.86%,real
8,91342,0644000,Los Angeles,Los Angeles County,1/31/2018,"$452,365",2018,"$58,385","$33,420",19.05%,7.05%,3959657.0,0.25%,35.4,33.71%,11.63%,1474043.0,1.12%,6.8%,36.84%,30.53%,38.21%,"$599,700","$1,376",23.74%,real
9,91911,0613392,Chula Vista,San Diego County,1/31/2018,"$451,680",2018,"$76,354","$29,143",10.85%,9.04%,266468.0,0.9%,34.9,28.75%,8.85%,85810.0,0.45%,8.01%,58.35%,4.05%,53.39%,"$465,000","$1,536",18.74%,real
